# einops-reduce composite — cx11: sum-broadcast duality via einops.reduce 'sum' as the back-fn

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-reduce`, `sum-and-broadcast-duality`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "einops-reduce"
DD_ATOM_IDS = ["einops-reduce", "sum-and-broadcast-duality"]
DD_SUBTOPICS = ["Einops: Reduce", "Backprop: sum/broadcast duality"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Forward `sum(x, dim)` is a reduce; its backward expands the upstream grad. Forward `broadcast_to(x, shape)` is an expand; its backward sums out the expanded axes. The two are exact adjoints — the sum/broadcast duality.

The trick: you don't need a hand-rolled `unbroadcast` to write `broadcast_back`. `einops.reduce` with `'sum'` IS the unbroadcast. If forward expanded `(R,)` to `(R, C)` (broadcast_to introduces or stretches axes), then `broadcast_back` is `reduce(grad_out, 'r c -> r', 'sum')`. And if forward summed `(R, C) -> (R,)`, then `sum_back` re-broadcasts via `repeat(grad_out, 'r -> r c', c=C)`. The reduce atom and the duality atom collapse into the same einops vocabulary.

### Composite Exercise — sum-broadcast duality via einops.reduce 'sum' as the back-fn

**Atoms exercised together**: `einops-reduce`, `sum-and-broadcast-duality`

Implement TWO functions wired together, then sanity-check they round-trip:

1. `cx11_sum_forward(x)` — forward pass: collapse the column axis of `(R, C)` to `(R,)` using `einops.reduce(x, 'r c -> r', 'sum')`. Return the result.
2. `cx11_broadcast_back(grad_out, x)` — backward of a `(R,) -> (R, C)` broadcast (equivalently, the gradient w.r.t. the size-1 / absent axis when forward broadcast). Implement it as `einops.reduce(grad_out, 'r c -> r', 'sum')` — exact same call pattern as the sum forward. The duality says: backward of broadcast IS reduce-with-sum.

Round-trip invariant: if forward was `out = x.sum(dim=1)` with grad_out `ones_like(out)`, then `broadcast_back(ones_like(x), x)` must equal `ones_like(out)` (each column contributes 1 unit of grad → the row total is C ones, but the GRAD for the original `(R,)` shape — i.e. the broadcast input — sums those C ones back into a scalar per row). More precisely, calling `broadcast_back(t.ones_like(x), x)` returns a `(R,)` tensor of all-C.

In [ ]:
def cx11_sum_forward(x):
    # Atom: einops.reduce — collapse C with 'sum'.
    return reduce(x, 'r c -> r', 'sum')

def cx11_broadcast_back(grad_out, x):
    # Atom: sum/broadcast duality — backward of broadcast IS reduce-with-'sum'.
    # Same einops call pattern as the forward sum; that IS the duality.
    return reduce(grad_out, 'r c -> r', 'sum')


<details><summary>Show solution — cx11</summary>

```python
def cx11_sum_forward(x):
    # Atom: einops.reduce — collapse C with 'sum'.
    return reduce(x, 'r c -> r', 'sum')

def cx11_broadcast_back(grad_out, x):
    # Atom: sum/broadcast duality — backward of broadcast IS reduce-with-'sum'.
    # Same einops call pattern as the forward sum; that IS the duality.
    return reduce(grad_out, 'r c -> r', 'sum')
```

The point of the composite is to make the duality concrete: when both forward sum and the backward of broadcast share the SAME einops.reduce('sum') call, the adjoint relationship is no longer an abstract claim — it's the same line of code.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx11'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx11',
        'subtopics': ["Einops: Reduce", "Backprop: sum/broadcast duality"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()